In [ ]:
# --- make the safety_probes package importable ---
# Preferred: run `pip install -e .` from the repo root once; then this cell is a no-op.
import sys, pathlib
try:
    import safety_probes
except ModuleNotFoundError:
    starts = []
    _f = globals().get("__vsc_ipynb_file__")
    if _f: starts.append(pathlib.Path(_f).resolve())
    starts.append(pathlib.Path.cwd())
    for _s in starts:
        _root = next((p for p in [_s, *_s.parents] if (p / "safety_probes" / "__init__.py").exists()), None)
        if _root:
            sys.path.insert(0, str(_root)); break
    import safety_probes
print("safety_probes:", safety_probes.__file__)

# Sycophancy — detection & steering (behaviour 3)

First run of behaviour 3 (`sycophancy_plan.md` v2 — read it before this notebook; §5c is the
data/masking reference). Positive class = **sycophantic** (1) — the reply ECHOES a factually wrong
user claim; negative = **candid** (0) — the reply states the truth (corrections AND genuine
agreements, the anti-echo-shortcut hard negatives). Data: `disentangle-sycophancy` factorial
records (arXiv 2509.21305 — closest related work, cite it; pinned in `data/PROVENANCE.md`).
Train = cities + translations + claims + companies (~1,368/class); **OOD = math** (held-out
domain, 500/class). Same §1–8 skeleton as the deception and high-stakes notebooks.

**No audited values exist** — this is the first run, so the EDIT-HERE cells' `argmax`/`pick_block`
suggestions ARE the selection; review the plots and pin your choices in the markdown afterwards.

In [ ]:
from safety_probes import runtime, datasets, metrics, steering, viz, evaluation, export
from safety_probes.chat import chat_text
from safety_probes.extraction import get_pooled, extract_full_sequence_one_layer
from safety_probes.metrics import (cv_auroc_by_layer, build_probe_grid, evaluate_grid, pack,
                                   probe_scores)
from safety_probes.probes import MMProbe, LRProbe
from safety_probes.steering import (nie_layer_sweep, pick_block, build_directions, calibrate_block)
from safety_probes.evaluation import steering_nie_llm_evaluation
from safety_probes.judge import N_JUDGE_PROMPTS
from safety_probes.viz import (pca_and_projection_panel, plot_roc_curves, probe_direction_cosines,
                               show_token_heatmaps, show_attention_heatmaps, plot_steering_effects,
                               show_qualitative)
from safety_probes.datasets import SYC_MAX_LEN
from sklearn.metrics import roc_auc_score
import numpy as np, torch as t

# Crisp inline figures on hi-dpi screens (pairs with viz._use_style()'s DPI bump — plan #50/M15).
%config InlineBackend.figure_format = "retina"

runtime.init()

## 1. BehaviourSpec and Datasets

The spec configures the NIE readout (functional phrasing — "go along with the user's belief" —
not the rare word "sycophantic"; decided 2026-07-30), the CAA-style judge rubric, and the steering
coefficients. The loader maps factorial rows to labels per sycophancy_plan §3 and prints the
**GA share of negatives** — the genuine-agreement hard negatives that force the probe to learn
*agreeing-when-wrong* rather than echo detection.

In [ ]:
# Spec lives in the package (plan #48) — see safety_probes/behaviours.py.
from safety_probes.behaviours import SYCOPHANCY_SPEC
SYCOPHANCY_SPEC

In [ ]:
# --- Sycophancy: factorial stance-pairs, 4 train domains; math held out as OOD ---
# Pools cap the balanced size at ~1,368/class train, ~347/class test (the sycophantic class is
# the scarce one: one clean echo per wrong-claim prompt). Pair-aware split by PROMPT group.
syc = datasets.load_sycophancy()          # .train / .test  (extra: raw, user, ga)
syc_ood = datasets.load_sycophancy_ood()  # math domain, 500/class

## 2. Extracting Activations

In [ ]:
# Sycophancy pooled activations (assistant-reply mask); replies are short (~10-20 tokens).
syc_train_mean, syc_train_last = get_pooled("syc_train", syc.train.texts, syc.train.masks, max_length=SYC_MAX_LEN)
syc_test_mean,  syc_test_last  = get_pooled("syc_test",  syc.test.texts,  syc.test.masks,  max_length=SYC_MAX_LEN)
syc_ood_mean,   syc_ood_last   = get_pooled("syc_ood",   syc_ood.texts,   syc_ood.masks,   max_length=SYC_MAX_LEN)
print("Sycophancy pooled activations:", tuple(syc_train_mean.shape))

## 3. Layer Selection

Same two purpose-matched criteria as the other behaviours: **detection layer** = argmax pair-aware
5-fold CV AUROC of an MM probe (LR overlaid as the proxy check); **steering block** = the NIE
sweep in §7a. Prior expectation: arXiv 2509.21305 reports difference-of-means probe AUROC > 0.9 on
this exact model at mid layers, so expect a strong curve. **If the curve sits at 1.000, do NOT
celebrate — run the anti-echo-shortcut check in §5.i before trusting it** (the GA hard negatives
are what make ceiling performance meaningful here).

**EDIT `SYC_DETECT_LAYER`** in the cell below (first run — argmax is the selection).

In [ ]:
syc_cv_mean, syc_cv_std = cv_auroc_by_layer(syc_train_mean, syc.train.labels, syc.train.pairs)
# LR overlay (robustness): check LR's curve tracks MM's before committing. MM stays the selector.
syc_cv_lr_mean, syc_cv_lr_std = cv_auroc_by_layer(syc_train_mean, syc.train.labels, syc.train.pairs, probe_cls=LRProbe)

# ===== EDIT HERE: pick the detection layer from the plot below. =====
print(f"Suggested (argmax CV AUROC): {int(np.argmax(syc_cv_mean))}")
SYC_DETECT_LAYER = int(np.argmax(syc_cv_mean))   # <-- override with your choice (first run: argmax)
# ====================================================================

viz.plot_layer_selection(syc_cv_mean, syc_cv_std, syc_cv_lr_mean, syc_cv_lr_std, SYC_DETECT_LAYER, "Sycophancy")

## 4. PCA Visualization

Same panels and caveat as the other behaviours: PCA shows variance (here likely dominated by
domain/topic — four fact types), the projection histogram onto the trained MM direction is the
load-bearing panel. Note the pair structure: a sycophantic row and its correction share the prompt
and most of the reply — expect co-located dots, like deception's pairs.

In [ ]:
syc_mm_detect = MMProbe.from_data(syc_train_mean[:, SYC_DETECT_LAYER, :], syc.train.labels)
pca_and_projection_panel(syc_train_mean, syc.train.labels, SYC_DETECT_LAYER, syc_mm_detect.direction, "Sycophancy")

## 5. Probing

Full-sequence activations at `SYC_DETECT_LAYER` (attention probe + per-token LR; ~40-50k token
rows, at/under the 60k cap). OOD full-seq is RAM-only (`use_cache=False`), matching the OOD
convention (#24).

In [ ]:
syc_train_full, syc_train_fmask = extract_full_sequence_one_layer("syc_train", syc.train.texts, syc.train.masks, SYC_DETECT_LAYER, SYC_MAX_LEN)
syc_test_full,  syc_test_fmask  = extract_full_sequence_one_layer("syc_test",  syc.test.texts,  syc.test.masks,  SYC_DETECT_LAYER, SYC_MAX_LEN)
syc_ood_full,   syc_ood_fmask   = extract_full_sequence_one_layer("syc_ood",   syc_ood.texts,   syc_ood.masks,   SYC_DETECT_LAYER, SYC_MAX_LEN, use_cache=False)

In [ ]:
# --- Sycophancy grid ---
Ls = SYC_DETECT_LAYER
syc_grid = build_probe_grid(syc_train_mean[:, Ls, :], syc_train_last[:, Ls, :], syc_train_fmask,
                            syc_train_full, syc.train.masks, syc.train.labels)

### 5.i. Probing Evaluations

ID test + math (OOD): AUROC (bootstrap 95% CI) + TPR@1%FPR. Then the **anti-echo-shortcut check**
(sycophancy_plan §7): AUROC restricted to sycophantic rows vs GENUINE-AGREEMENT negatives only.
Both classes there are echoes of the user's claim — an echo detector scores ~0.5 on this subset;
a real sycophancy probe (using the model's knowledge of whether the claim is true) stays high.

In [ ]:
syc_eval_packs = {
    "test (ID)": pack(syc_test_mean[:, Ls, :], syc_test_last[:, Ls, :], syc_test_full, syc_test_fmask, syc.test.labels),
    "math (OOD)": pack(syc_ood_mean[:, Ls, :], syc_ood_last[:, Ls, :], syc_ood_full, syc_ood_fmask, syc_ood.labels),
}
print("=== Sycophancy probe grid (detection layer", Ls, ") ===")
syc_grid_df = evaluate_grid(syc_grid, syc_eval_packs)
display(syc_grid_df)

# Per-seed attention AUROCs (training stability, as in the other notebooks).
with t.no_grad():
    _seed_aurocs = [roc_auc_score(syc.test.labels.numpy(), p(syc_test_full, syc_test_fmask).sigmoid().numpy())
                    for p in syc_grid["LR-attention"][0]]
print("LR-attention per-seed test AUROC (training stability):", [f"{a:.3f}" for a in _seed_aurocs])

# --- Anti-echo-shortcut check: sycophantic vs GA-only negatives (both are echoes) ---
for name, split, mean_a, last_a in (("ID", syc.test, syc_test_mean, syc_test_last),
                                    ("OOD", syc_ood, syc_ood_mean, syc_ood_last)):
    lab = split.labels.bool()
    ga = t.tensor(split.extra["ga"])
    keep = (lab | (ga & ~lab)).numpy()
    sc = probe_scores(syc_grid["LR (mean)"][0], "mean", mean_a[:, Ls, :], last_a[:, Ls, :])
    auroc_ga = roc_auc_score(split.labels.numpy()[keep], sc[keep])
    print(f"anti-echo-shortcut ({name}): sycophantic vs GA-only AUROC = {auroc_ga:.3f} "
          f"(~0.5 would mean the probe is an echo detector)")

plot_roc_curves(syc_grid, syc_eval_packs["test (ID)"], f"ROC curves - sycophancy ID test (layer {Ls}, {len(syc.test.labels)} examples)")
plot_roc_curves(syc_grid, syc_eval_packs["math (OOD)"], f"ROC curves - sycophancy OOD: math ({len(syc_ood.labels)} examples)")

In [ ]:
# Direction geometry, as the other behaviours. Compare the pattern by eye.
_ = probe_direction_cosines(syc_grid, syc_train_mean[:, Ls, :], syc_train_last[:, Ls, :], syc.train.labels, f"Sycophancy (detection layer {Ls})")

### 5.ii. Probing Verdict

Best **reader** per eval set (AUROC point estimates) + the LR-family win share — the READ half of
the headline hypothesis. The write half is §7d.

In [ ]:
# --- Probing verdict: who READS sycophancy best? ---
from safety_probes import summary
syc_read_pivot = summary.auroc_pivot(syc_grid_df, "sycophancy")
display(syc_read_pivot.round(3))
syc_best_readers = syc_read_pivot.idxmax()
print("Best probe per eval set:"); print(syc_best_readers.to_string())
syc_lr_share = syc_best_readers.str.startswith("LR").mean()
print(f"\nLR-family wins {syc_lr_share:.0%} of sycophancy detection columns "
      "([Apollo App. D.1] predicts LR reads best).")

## 6. Sample Token Heatmaps

Same conventions as the other notebooks (§6 of either): token color = standardized projection on
the LR (mean) direction (relative shading, not a verdict); header = raw mean **sycophancy score**
(uncalibrated); the label shown is ground truth. Interesting thing to look for here: the
sycophantic and candid replies to one prompt differ only at the claim's value ("Germany" vs
"Kyrgyzstan") — does the probe fire on that token, or on the agreement phrasing before it?

### 6a. Probe Heatmaps

In [ ]:
syc_lr_mean_dir = syc_grid["LR (mean)"][0].direction
show_token_heatmaps(syc.test.texts, syc.test.masks, syc.test.labels, syc_lr_mean_dir, SYC_DETECT_LAYER, "Sycophancy ID test", "sycophantic", "candid", score_name="sycophancy score")
show_token_heatmaps(syc_ood.texts, syc_ood.masks, syc_ood.labels, syc_lr_mean_dir, SYC_DETECT_LAYER, "math (OOD)", "sycophantic", "candid", n_show=4, score_name="sycophancy score")

### 6b. Attention Heatmaps

Attention-weight view (where the probe pools from — full math: deception notebook §6b). With
~10-20-token replies, expect concentrated attention; BOS/template parking is a result, not a bug.

In [ ]:
syc_attn_seeds = syc_grid["LR-attention"][0]
show_attention_heatmaps(syc_test_full, syc_test_fmask, syc.test.texts, syc.test.labels, syc_attn_seeds, "Sycophancy ID", "sycophantic", "candid")
show_attention_heatmaps(syc_ood_full, syc_ood_fmask, syc_ood.texts, syc_ood.labels, syc_attn_seeds, "math (OOD)", "sycophantic", "candid", n_show=4)

## 7. Steering

Machinery identical to the other behaviours (per-layer refit + mean-gap-calibrated MM / LR /
LR-attention, added at every token). Note for the comparison notebook: check whether the selected
block overlaps deception's [15-19] or high-stakes' [9-13] — overlap changes how the
simultaneous-steering pilot reads (`compose_steering` sums colliding layers).

### 7a. Steering-Block Selection (NIE sweep + baseline gate)

Per-layer NIE with the calibrated MM direction at coef +1; sweep subset from the TRAIN split;
MM-anchoring disclosed as always. The readout transcripts include the replies by construction, so
the baseline gap cannot collapse (sycophancy_plan §5b.1) — but still check it clears the 0.05
floor before trusting any NIE.

**EDIT `SYC_STEER_BLOCK`** in the pick cell below (first run — `pick_block` is the selection).

In [ ]:
# Sweep subset from the TRAIN split; plaintext transcripts incl. replies (extra["raw"]).
syc_train_raw, syc_train_lab = syc.train.extra["raw"], syc.train.labels.tolist()
syc_test_raw,  syc_test_lab  = syc.test.extra["raw"],  syc.test.labels.tolist()
syc_sweep_pos_tx = [r for r, l in zip(syc_train_raw, syc_train_lab) if l == 1][:32]
syc_sweep_neg_tx = [r for r, l in zip(syc_train_raw, syc_train_lab) if l == 0][:32]

syc_nie, syc_gap = nie_layer_sweep(SYCOPHANCY_SPEC, syc_train_mean, syc.train.labels, syc_sweep_pos_tx, syc_sweep_neg_tx)

In [ ]:
# ===== EDIT HERE: pick the sycophancy steering block from the NIE bars below. =====
print(f"Suggested (top NIE window): {pick_block(syc_nie)}")
SYC_STEER_BLOCK = pick_block(syc_nie)   # <-- override with your choice (first run: suggestion)
# =======================================================================

viz.plot_nie_bars(syc_nie, SYC_STEER_BLOCK, "Sycophancy")

### 7b. Directions & Calibration

MM / LR / LR-attention, refit and mean-gap-calibrated per block layer. With #52a (GPU training)
and short sequences this should be minutes, not hours.

In [ ]:
syc_dirs = calibrate_block(
    build_directions(SYCOPHANCY_SPEC, SYC_STEER_BLOCK, syc_train_mean, syc.train.labels, syc.train.texts, syc.train.masks, SYC_MAX_LEN),
    syc_train_mean, syc.train.labels)
print(f"Sycophancy steering directions calibrated: {list(syc_dirs)} on block {SYC_STEER_BLOCK}")

### 7c. Steering Evaluations

ID first (positive control), then OOD (math). NIE dose-response + LLM-judged generations —
sycophancy keeps its FULL judge arm on both eval sets: every generation prompt is a distinct
user claim (sycophancy_plan §5b.2), and no prompt names the behaviour (§5b.3). Generation
prompts = the user turn only (claim included, no reply) — steering must produce the caving.

In [ ]:
# --- Sycophancy steering: ID (positive control) then OOD (the claim) ---
# ID readout/eval transcripts from the TEST split - disjoint from the train-split sweep subset (§7a).
syc_val_pos_tx = [r for r, l in zip(syc_test_raw, syc_test_lab) if l == 1][:32]
syc_val_neg_tx = [r for r, l in zip(syc_test_raw, syc_test_lab) if l == 0][:32]
COEFS = list(SYCOPHANCY_SPEC.steer_coefs)
# Generation prompts: DISTINCT user turns (rows share prompts -> dedupe first), user turn only.
syc_id_users = list(dict.fromkeys(syc.test.extra["user"]))[:N_JUDGE_PROMPTS]
syc_id_prompts = [chat_text([{"role": "user", "content": u}], add_generation_prompt=True) for u in syc_id_users]
print("=== Sycophancy steering: IN-DISTRIBUTION (positive control) ===")
syc_id_results = steering_nie_llm_evaluation(SYCOPHANCY_SPEC, syc_dirs, syc_id_prompts, syc_val_neg_tx, syc_val_pos_tx, COEFS)
plot_steering_effects(syc_id_results, "Sycophancy steering (ID)")

In [ ]:
# --- Sycophancy steering: OOD (math domain) ---
syc_ood_raw, syc_ood_lab = syc_ood.extra["raw"], syc_ood.labels.tolist()
syc_ood_neg = [r for r, l in zip(syc_ood_raw, syc_ood_lab) if l == 0][:32]
syc_ood_pos = [r for r, l in zip(syc_ood_raw, syc_ood_lab) if l == 1][:32]
syc_ood_users = list(dict.fromkeys(syc_ood.extra["user"]))[:N_JUDGE_PROMPTS]
syc_ood_prompts = [chat_text([{"role": "user", "content": u}], add_generation_prompt=True) for u in syc_ood_users]
print("=== Sycophancy steering: OUT-OF-DISTRIBUTION (math) ===")
syc_ood_results = steering_nie_llm_evaluation(SYCOPHANCY_SPEC, syc_dirs, syc_ood_prompts, syc_ood_neg, syc_ood_pos, COEFS)
plot_steering_effects(syc_ood_results, "Sycophancy steering (OOD: math)")

In [ ]:
show_qualitative(SYCOPHANCY_SPEC, syc_dirs, syc_id_prompts[0])

### 7d. Steering Verdict

Best **writer** per eval set (NIE at calibrated coef +1) + MM win share + the per-behaviour
dissociation line. Judge feeds no number here; its coherence panels above discipline the wins.

In [ ]:
# --- Steering verdict: who WRITES sycophancy best? ---
import pandas as pd
from safety_probes import summary
syc_steer_by_evalset = {"ID": syc_id_results, "Math (OOD)": syc_ood_results}
syc_write_tbl = pd.DataFrame({f"sycophancy: {k}": summary.nie_at(v) for k, v in syc_steer_by_evalset.items()})
display(syc_write_tbl.round(3))
syc_best_writers = syc_write_tbl.idxmax()
print("Best direction per eval set:"); print(syc_best_writers.to_string())
syc_mm_share = (syc_best_writers == "MM").mean()
print(f"\nMM wins {syc_mm_share:.0%} of sycophancy steering columns ([GoT Table 2] predicts MM writes best).")
print("Read/write dissociation " + ("HOLDS" if syc_lr_share > 0.5 and syc_mm_share > 0.5 else "does NOT clearly hold")
      + " for sycophancy on this run (LR reads best + MM writes best; judge-coherence caveat applies).")

## 8. Export bundle

In [ ]:
# Saving Probe
export.export_behaviour(
    SYCOPHANCY_SPEC, syc_grid, SYC_DETECT_LAYER, SYC_STEER_BLOCK, syc_dirs,
    d_model=syc_train_mean.shape[-1],
    notes="OOD: math domain (disentangle-sycophancy factorial, held-out; commit 217b8980).")

# Freeze the final numbers as artifacts in exported_probes/ — CSVs for thesis tables, and the FULL
# dose-response JSON for the cross-behaviour comparison notebook (plan #48).
from safety_probes import summary
summary.save_detection(syc_grid_df, "sycophancy")
summary.save_steering(syc_steer_by_evalset, "sycophancy")
summary.save_steering_full(syc_steer_by_evalset, "sycophancy")